In [1]:
!pip install -q kaggle timm albumentations tqdm opencv-python-headless seaborn

import os, zipfile, torch
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    f.write('{"username":"ahnafnafim","key":"KGAT_501049cf57c12a07edf49389e126d5cc"}')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

data_dir = Path('/tmp/data')
data_dir.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d araftahsanpavel/tgif-subset -p /tmp/data/ -q
with zipfile.ZipFile(data_dir / 'tgif-subset.zip', 'r') as z:
    z.extractall(data_dir)
(data_dir / 'tgif-subset.zip').unlink(missing_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print("✓ Done")

Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
GPU: NVIDIA A100 80GB PCIe
✓ Done


In [10]:
from __future__ import annotations
import os, time, random, json, copy
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm

IMAGE_SIZE          = 384
ABLATION_EPOCHS     = 15
BATCH_SIZE          = 8
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-4
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
LR_MIN              = 1e-6
EARLY_STOP_PATIENCE = 7
BINARY_THRESHOLD    = 0.5
GRAD_CLIP_NORM      = 1.0
MIXED_PRECISION     = True
SEED                = 42
NUM_WORKERS         = 4
PIN_MEMORY          = True
POS_WEIGHT          = 3.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
DATA_ROOT  = Path('/tmp/data/subset')
SAVE_DIR   = Path('/home/ubuntu/results_base_ablation')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False

print("=" * 70)
print("  ConvNeXt-L Base Ablation — Building up to ForensicNet")
print("=" * 70)
print(f"  GPU        : {torch.cuda.get_device_name(0)}")
print(f"  IMAGE_SIZE : {IMAGE_SIZE}")
print(f"  EPOCHS     : {ABLATION_EPOCHS}")
print(f"  SAVE_DIR   : {SAVE_DIR}")
print("=" * 70)

  ConvNeXt-L Base Ablation — Building up to ForensicNet
  GPU        : NVIDIA A100 80GB PCIe
  IMAGE_SIZE : 384
  EPOCHS     : 15
  SAVE_DIR   : /home/ubuntu/results_base_ablation


In [2]:
def infer_mask_path_from_fake(fake_path: Path, masks_root: Path) -> Optional[Path]:
    category = fake_path.parent.name
    name = fake_path.name
    if '_mask_' not in name:
        return None
    stem = name
    for tag in ['_sd2_', '_sdxl_', '_firefly_', '_dalle_', '_dalle2_']:
        if tag in stem:
            stem = stem.split(tag)[0] + '.png'
            break
    candidate = masks_root / category / stem
    if candidate.exists():
        return candidate
    mask_dir = masks_root / category
    if mask_dir.exists():
        for m in mask_dir.glob('*.png'):
            if name.startswith(m.name):
                return m
    return None


def build_index(split_dir: Path) -> List[Dict[str, Any]]:
    images_root = split_dir / 'images'
    masks_root  = split_dir / 'masks'
    fakes_root  = split_dir / 'fakes'
    samples = []
    if images_root.exists():
        for p in images_root.rglob('*_orig.png'):
            samples.append({'image_path': str(p), 'mask_path': None,
                           'is_fake': False, 'category': p.parent.name})
    if fakes_root.exists():
        for p in fakes_root.rglob('*.png'):
            if '_mask_' not in p.name:
                continue
            paired_mask = infer_mask_path_from_fake(p, masks_root)
            if paired_mask is None or not paired_mask.exists():
                continue
            samples.append({'image_path': str(p), 'mask_path': str(paired_mask),
                           'is_fake': True, 'category': p.parent.name})
    return samples


class TGIFDataset(Dataset):
    def __init__(self, samples, image_size, transform=None):
        self.samples = samples
        self.transform = transform or A.Compose([
            A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(s['image_path'], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if s['mask_path'] is not None:
            mask = (cv2.imread(s['mask_path'], cv2.IMREAD_GRAYSCALE) >= 127).astype(np.uint8)
        else:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        out = self.transform(image=img, mask=mask)
        return {
            'image': out['image'].float(),
            'mask': out['mask'].float().unsqueeze(0),
            'is_fake': torch.tensor(s['is_fake'], dtype=torch.bool),
        }


print("Indexing dataset ...")
train_samples = build_index(DATA_ROOT / 'training')
val_samples   = build_index(DATA_ROOT / 'validation')
test_samples  = build_index(DATA_ROOT / 'testing')

for name, samples in [('training', train_samples), ('validation', val_samples), ('testing', test_samples)]:
    n_fake = sum(1 for s in samples if s['is_fake'])
    print(f"  {name:<10}: {len(samples):>5} total  ({len(samples)-n_fake:>4} real, {n_fake:>4} fake)")

# No augmentation transforms
noaug_tf = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# Augmentation transforms
aug_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0, p=0.4),
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# Default loaders (no augmentation)
train_loader_noaug = DataLoader(TGIFDataset(train_samples, IMAGE_SIZE, noaug_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
train_loader_aug = DataLoader(TGIFDataset(train_samples, IMAGE_SIZE, aug_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(TGIFDataset(val_samples, IMAGE_SIZE, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(TGIFDataset(test_samples, IMAGE_SIZE, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"  Train (noaug): {len(train_loader_noaug)} batches")
print(f"  Train (aug):   {len(train_loader_aug)} batches")
print(f"  Val: {len(val_loader)} | Test: {len(test_loader)}")

Indexing dataset ...


  training  :  7559 total  (2100 real, 5459 fake)
  validation:  1023 total  ( 341 real,  682 fake)
  testing   :  1029 total  ( 343 real,  686 fake)
  Train (noaug): 945 batches
  Train (aug):   945 batches
  Val: 128 | Test: 129


In [4]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        inter = (probs * targets).sum(dim=1)
        denom = probs.sum(dim=1) + targets.sum(dim=1)
        return 1.0 - ((2.0 * inter + self.smooth) / (denom + self.smooth)).mean()


class MetricAccumulator:
    def __init__(self):
        self.tp = self.fp = self.tn = self.fn = 0
    def update(self, preds_binary, targets):
        p = preds_binary.bool(); t = targets.bool()
        self.tp += int((p & t).sum().item())
        self.fp += int((p & ~t).sum().item())
        self.tn += int((~p & ~t).sum().item())
        self.fn += int((~p & t).sum().item())
    def compute(self):
        eps = 1e-8; tp, fp, tn, fn = self.tp, self.fp, self.tn, self.fn
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        return {
            'pixel_accuracy': (tp + tn) / (tp + fp + tn + fn + eps),
            'precision': precision, 'recall': recall, 'f1': f1,
            'iou': tp / (tp + fp + fn + eps),
            'dice': 2 * tp / (2 * tp + fp + fn + eps),
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        }


class BCEDiceLoss(nn.Module):
    """Simple seg-only loss for base ablations."""
    def __init__(self, pos_weight=POS_WEIGHT):
        super().__init__()
        self.register_buffer('pw', torch.tensor([pos_weight], dtype=torch.float32))
        self.dice = DiceLoss()

    def forward(self, outputs, masks, labels=None):
        logits = outputs['mask'] if isinstance(outputs, dict) else outputs
        bce = F.binary_cross_entropy_with_logits(logits, masks, pos_weight=self.pw.to(logits.device))
        total = 0.5 * bce + 0.5 * self.dice(logits, masks)
        return {'total': total, 'seg': total.detach()}

print("✓ Metrics + loss ready")

✓ Metrics + loss ready


In [5]:
# ═══════════════════════════════════════════════════════════════
# Shared components
# ═══════════════════════════════════════════════════════════════

class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        f1 = np.array([[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]], dtype=np.float32) / 4.0
        f2 = np.array([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]], dtype=np.float32) / 12.0
        f3 = np.array([[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=np.float32) / 2.0
        self.register_buffer('weight', torch.from_numpy(np.stack([f1, f2, f3])[:, None]))
    def forward(self, x):
        return torch.cat([F.conv2d(x[:, c:c+1], self.weight, padding=2) for c in range(3)], dim=1)


class BayarConv2d(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, k=5):
        super().__init__(); self.k = k
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.01)
    def forward(self, x):
        w = self.weight.clone(); c = self.k // 2
        wn = w.clone(); wn[:, :, c, c] = 0.0
        w[:, :, c, c] = -wn.sum(dim=[2, 3])
        return F.conv2d(x, w, padding=c)


class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__(); mid = max(ch // r, 8)
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(ch, mid), nn.ReLU(),
                                nn.Linear(mid, ch), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = x * self.ca(x).view(x.size(0), -1, 1, 1)
        return x * self.sa(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1))


class NoiseBranch(nn.Module):
    def __init__(self, out_ch=128):
        super().__init__()
        self.srm = SRMConv(); self.bayar = BayarConv2d(3, 3)
        self.enc = nn.Sequential(
            nn.Conv2d(12, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, out_ch, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x):
        return self.enc(torch.cat([self.srm(x), self.bayar(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
# VARIANT 1: ConvNeXt-L + UNet (vanilla decoder, no attention)
# ═══════════════════════════════════════════════════════════════

class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# ═══════════════════════════════════════════════════════════════
# VARIANT 2: FPN decoder block (no CBAM)
# ═══════════════════════════════════════════════════════════════

class FPNBlockNoCBAM(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# ═══════════════════════════════════════════════════════════════
# VARIANT 3: FPN decoder block WITH CBAM (same as ForensicNet)
# ═══════════════════════════════════════════════════════════════

class FPNBlockWithCBAM(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU())
        self.cbam = CBAM(out_ch)
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))


# ═══════════════════════════════════════════════════════════════
# CONFIGURABLE MODEL — supports all ablation variants
# ═══════════════════════════════════════════════════════════════

class ConvNeXtLAblation(nn.Module):
    """
    Configurable ConvNeXt-L model for base ablation study.

    Configs:
      use_fpn:   False = UNet decoder, True = FPN lateral connections
      use_cbam:  CBAM attention in decoder (only with FPN)
      use_noise: SRM + Bayar noise branch
    """
    def __init__(self, config, image_size=IMAGE_SIZE, pretrained=True):
        super().__init__()
        self.image_size = image_size
        self.config = config

        # ConvNeXt-L encoder: [192, 384, 768, 1536]
        self.enc = timm.create_model('convnext_large', pretrained=pretrained,
                                      features_only=True, out_indices=(0, 1, 2, 3))

        # Noise branch (optional)
        if config['use_noise']:
            self.noise = NoiseBranch(out_ch=128)
            bottleneck_ch = 1536 + 128
        else:
            self.noise = None
            bottleneck_ch = 1536

        # Bottleneck projection
        self.lat4 = nn.Conv2d(bottleneck_ch, 256, 1)

        # Decoder variant
        if config['use_fpn']:
            self.lat3 = nn.Conv2d(768, 256, 1)
            self.lat2 = nn.Conv2d(384, 128, 1)
            self.lat1 = nn.Conv2d(192, 64, 1)

            if config['use_cbam']:
                Block = FPNBlockWithCBAM
            else:
                Block = FPNBlockNoCBAM

            self.dec3 = Block(256, 256, 256)
            self.dec2 = Block(256, 128, 128)
            self.dec1 = Block(128, 64, 64)
            self.dec0 = Block(64, 0, 32)
        else:
            # UNet vanilla decoder
            self.dec3 = UNetDecoderBlock(256, 768, 256)
            self.dec2 = UNetDecoderBlock(256, 384, 128)
            self.dec1 = UNetDecoderBlock(128, 192, 64)
            self.dec0 = UNetDecoderBlock(64, 0, 32)

        # Segmentation head
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16, 1, 1))

    def forward(self, x):
        s1, s2, s3, s4 = self.enc(x)

        # Noise fusion
        if self.noise is not None:
            n_feat = F.adaptive_avg_pool2d(self.noise(x), s4.shape[2:])
            p4 = self.lat4(torch.cat([s4, n_feat], dim=1))
        else:
            p4 = self.lat4(s4)

        # Decode
        if self.config['use_fpn']:
            d3 = self.dec3(p4, self.lat3(s3))
            d2 = self.dec2(d3, self.lat2(s2))
            d1 = self.dec1(d2, self.lat1(s1))
        else:
            d3 = self.dec3(p4, s3)
            d2 = self.dec2(d3, s2)
            d1 = self.dec1(d2, s1)

        d0 = self.dec0(d1)

        seg_out = self.head(d0)
        if seg_out.shape[2:] != (self.image_size, self.image_size):
            seg_out = F.interpolate(seg_out, size=(self.image_size, self.image_size),
                                     mode='bilinear', align_corners=False)

        return {'mask': seg_out}


# Verify all variants
#for name, cfg in [
  #  ('UNet vanilla',    {'use_fpn': False, 'use_cbam': False, 'use_noise': False}),
   # ('FPN',             {'use_fpn': True,  'use_cbam': False, 'use_noise': False}),
  #  ('FPN + CBAM',      {'use_fpn': True,  'use_cbam': True,  'use_noise': False}),
  #  ('FPN + CBAM + Noise', {'use_fpn': True,  'use_cbam': True,  'use_noise': True}),
#]:
   # with torch.no_grad():
        m = ConvNeXtLAblation(cfg, pretrained=False)
        o = m(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE))
        p = sum(pp.numel() for pp in m.parameters())
        print(f"  ✓ {name:<20}: {o['mask'].shape} | {p/1e6:.1f}M params")
        del m, o
#print("✓ All variants verified")

In [6]:
def run_one_epoch(model, loader, criterion, optimizer, scaler, device, train, desc):
    model.train(mode=train); total_loss = 0.0; n_batches = 0
    metrics = MetricAccumulator()
    grad_ctx = torch.enable_grad() if train else torch.no_grad()

    with grad_ctx:
        pbar = tqdm(loader, desc=desc, leave=False)
        for batch in pbar:
            images = batch['image'].to(device, non_blocking=True)
            targets = batch['mask'].to(device, non_blocking=True)
            labels = batch['is_fake'].to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            amp_on = MIXED_PRECISION and device.type == 'cuda'
            with autocast(device_type='cuda', enabled=amp_on):
                outputs = model(images)
                loss_dict = criterion(outputs, targets, labels)
                loss = loss_dict['total']

            if train:
                if scaler and amp_on:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    optimizer.step()

            with torch.no_grad():
                metrics.update((torch.sigmoid(outputs['mask']) >= BINARY_THRESHOLD).float(), targets)

            total_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')

    return total_loss / max(n_batches, 1), metrics.compute()


def train_variant(config_name, config, epochs, train_loader, keep_checkpoint=False):
    print(f"\n{'='*70}")
    print(f"CONFIG: {config_name}")
    active = [k.replace('use_', '') for k, v in config.items() if v]
    print(f"Active: {active if active else ['none (vanilla)']} | Epochs: {epochs}")
    print(f"{'='*70}")

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    model = ConvNeXtLAblation(config, pretrained=True).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Params: {total_params/1e6:.1f}M | Size: {total_params*4/1024**2:.0f}MB")

    criterion = BCEDiceLoss(pos_weight=POS_WEIGHT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN)
    scaler = GradScaler(enabled=MIXED_PRECISION and device.type == 'cuda')

    best_val_iou = -1.0; best_epoch = 0; epochs_no_improve = 0; history = []
    ckpt_path = SAVE_DIR / f'best_{config_name}.pth'

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        train_loss, train_m = run_one_epoch(model, train_loader, criterion, optimizer,
                                             scaler, device, train=True,
                                             desc=f'{config_name} train {epoch}')
        val_loss, val_m = run_one_epoch(model, val_loader, criterion, None, None,
                                         device, train=False,
                                         desc=f'{config_name} val {epoch}')
        lr = optimizer.param_groups[0]['lr']
        epoch_secs = time.time() - epoch_start
        scheduler.step(val_m['iou'])

        history.append({
            'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
            'train_iou': train_m['iou'], 'val_iou': val_m['iou'],
            'train_f1': train_m['f1'], 'val_f1': val_m['f1'],
            'val_precision': val_m['precision'], 'val_recall': val_m['recall'],
            'lr': lr, 'epoch_seconds': epoch_secs})

        if val_m['iou'] > best_val_iou:
            best_val_iou = val_m['iou']; best_epoch = epoch; epochs_no_improve = 0
            torch.save({
                'model': model.state_dict(), 'epoch': epoch,
                'best_val_iou': best_val_iou, 'config': config,
                'config_name': config_name, 'history': history,
            }, ckpt_path)
            print(f"  Epoch {epoch:>3}/{epochs} | val IoU {val_m['iou']:.4f} | "
                  f"val F1 {val_m['f1']:.4f} | lr {lr:.2e} | {epoch_secs:.0f}s ✓ BEST")
        else:
            epochs_no_improve += 1
            if epoch % 5 == 0 or epochs_no_improve >= EARLY_STOP_PATIENCE - 1:
                print(f"  Epoch {epoch:>3}/{epochs} | val IoU {val_m['iou']:.4f} | "
                      f"val F1 {val_m['f1']:.4f} | lr {lr:.2e} | {epoch_secs:.0f}s | "
                      f"patience {epochs_no_improve}/{EARLY_STOP_PATIENCE}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"  [early stop] at epoch {epoch}"); break

    # Load best and evaluate test
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model'])

    test_loss, test_m = run_one_epoch(model, test_loader, criterion, None, None,
                                       device, train=False, desc=f'{config_name} test')

    print(f"\n  TEST — IoU: {test_m['iou']:.4f} | F1: {test_m['f1']:.4f} | "
          f"Prec: {test_m['precision']:.4f} | Rec: {test_m['recall']:.4f}")

    result = {
        'config_name': config_name, 'config': config,
        'epochs_trained': history[-1]['epoch'] if history else 0,
        'best_epoch': best_epoch, 'best_val_iou': best_val_iou,
        'test_metrics': test_m, 'history': history,
        'total_params': total_params,
        'model_size_mb': total_params * 4 / 1024**2,
        'checkpoint_path': str(ckpt_path) if keep_checkpoint and ckpt_path.exists() else None,
    }

    with open(SAVE_DIR / f'{config_name}_results.json', 'w') as f:
        json.dump(result, f, indent=2, default=str)

    if ckpt_path.exists() and not keep_checkpoint:
        os.remove(ckpt_path)
    elif ckpt_path.exists() and keep_checkpoint:
        print(f"  ✓ Kept checkpoint: {ckpt_path}")

    del model, criterion, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    return result

print("✓ Training engine ready")

✓ Training engine ready


In [7]:
import torch
import torch.nn as nn

torch.backends.cudnn.enabled = True
x = torch.randn(2, 128, 48, 48).cuda()
bn = nn.BatchNorm2d(128).cuda()
out = bn(x)
print(f"✓ cuDNN works | GPU: {torch.cuda.get_device_name(0)}")

✓ cuDNN works | GPU: NVIDIA A100 80GB PCIe


In [19]:
BASE_CONFIGS = {
    'base_1_convnextl_unet': {
        'config': {'use_fpn': False, 'use_cbam': False, 'use_noise': False},
        'keep_ckpt': True,  # Keep for XAI
    },
    'base_2_convnextl_fpn': {
        'config': {'use_fpn': True, 'use_cbam': False, 'use_noise': False},
        'keep_ckpt': False,
    },
    'base_3_convnextl_fpn_cbam': {
        'config': {'use_fpn': True, 'use_cbam': True, 'use_noise': False},
        'keep_ckpt': False,
    },
}

all_results = {}

print(f"Running {len(BASE_CONFIGS)} base ablation configs (no augmentation)...\n")

for name, spec in BASE_CONFIGS.items():
    result = train_variant(name, spec['config'], ABLATION_EPOCHS,
                           train_loader_noaug, keep_checkpoint=spec['keep_ckpt'])
    all_results[name] = result
    # Save partial results after each config
    with open(SAVE_DIR / 'base_ablation_partial.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"✓ Partial results saved after {name}")

print("\n✓ Base ablation complete!")

Running 3 base ablation configs (no augmentation)...


CONFIG: base_1_convnextl_unet
Active: ['none (vanilla)'] | Epochs: 15


Params: 200.7M | Size: 766MB


  Epoch   1/15 | val IoU 0.5727 | val F1 0.7283 | lr 1.00e-04 | 211s ✓ BEST


  Epoch   2/15 | val IoU 0.7080 | val F1 0.8290 | lr 1.00e-04 | 231s ✓ BEST


  Epoch   3/15 | val IoU 0.7611 | val F1 0.8643 | lr 1.00e-04 | 208s ✓ BEST


...

  Epoch  10/15 | val IoU 0.7856 | val F1 0.8799 | lr 1.00e-04 | 212s | patience 3/7


  Epoch  11/15 | val IoU 0.8211 | val F1 0.9018 | lr 1.00e-04 | 213s ✓ BEST


  Epoch  12/15 | val IoU 0.8288 | val F1 0.9064 | lr 1.00e-04 | 214s ✓ BEST


  Epoch  15/15 | val IoU 0.0000 | val F1 0.0000 | lr 1.00e-04 | 210s | patience 3/7



  TEST — IoU: 0.8228 | F1: 0.9028 | Prec: 0.8996 | Rec: 0.9060
✓ Partial results saved after base_3_convnextl_fpn_cbam

✓ Base ablation complete!


In [21]:
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import gc
gc.collect()
torch.cuda.empty_cache()
print(f"cuDNN re-enabled | Free GPU: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

cuDNN re-enabled | Free GPU: 54.6 GB


In [8]:
# Separate cell — run this if you want to reproduce ForensicNet baseline
print("=" * 70)
print("  OPTIONAL: ForensicNet Reproduction (ConvNeXt-L + FPN + CBAM + Noise)")
print("=" * 70)

forensicnet_config = {'use_fpn': True, 'use_cbam': True, 'use_noise': True}

result_forensicnet = train_variant(
    'base_4_forensicnet_reproduction',
    forensicnet_config,
    ABLATION_EPOCHS,
    train_loader_noaug,
    keep_checkpoint=True  # Keep for XAI comparison
)

all_results['base_4_forensicnet_reproduction'] = result_forensicnet

with open(SAVE_DIR / 'base_ablation_partial.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("✓ ForensicNet reproduction complete")

  OPTIONAL: ForensicNet Reproduction (ConvNeXt-L + FPN + CBAM + Noise)

CONFIG: base_4_forensicnet_reproduction
Active: ['fpn', 'cbam', 'noise'] | Epochs: 15
Params: 199.6M | Size: 761MB


  Epoch   1/15 | val IoU 0.6104 | val F1 0.7580 | lr 1.00e-04 | 452s ✓ BEST


  Epoch   2/15 | val IoU 0.6332 | val F1 0.7754 | lr 1.00e-04 | 440s ✓ BEST


  Epoch   3/15 | val IoU 0.7277 | val F1 0.8424 | lr 1.00e-04 | 436s ✓ BEST


  Epoch   5/15 | val IoU 0.7706 | val F1 0.8704 | lr 1.00e-04 | 443s ✓ BEST


...

  Epoch  12/15 | val IoU 0.8033 | val F1 0.8909 | lr 1.00e-04 | 429s ✓ BEST


  Epoch  14/15 | val IoU 0.8288 | val F1 0.9064 | lr 1.00e-04 | 438s ✓ BEST


  Epoch  15/15 | val IoU 0.0000 | val F1 0.0000 | lr 1.00e-04 | 431s | patience 1/7



  TEST — IoU: 0.8155 | F1: 0.8984 | Prec: 0.8945 | Rec: 0.9023
  ✓ Kept checkpoint: /home/ubuntu/results_base_ablation/best_base_4_forensicnet_reproduction.pth


NameError: name 'all_results' is not defined

In [11]:
print("=" * 70)
print("  ConvNeXt-L + UNet (vanilla) WITH AUGMENTATION")
print("=" * 70)

unet_aug_config = {'use_fpn': False, 'use_cbam': False, 'use_noise': False}

result_unet_aug = train_variant(
    'base_1_convnextl_unet_augmented',
    unet_aug_config,
    ABLATION_EPOCHS,
    train_loader_aug,  # <-- augmented dataloader
    keep_checkpoint=True  # Keep for XAI
)

all_results['base_1_convnextl_unet_augmented'] = result_unet_aug

with open(SAVE_DIR / 'base_ablation_partial.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("✓ Augmented UNet complete")

  ConvNeXt-L + UNet (vanilla) WITH AUGMENTATION

CONFIG: base_1_convnextl_unet_augmented
Active: ['none (vanilla)'] | Epochs: 15


Params: 200.7M | Size: 766MB


  Epoch   1/15 | val IoU 0.6096 | val F1 0.7574 | lr 1.00e-04 | 211s ✓ BEST


  Epoch   2/15 | val IoU 0.7004 | val F1 0.8238 | lr 1.00e-04 | 210s ✓ BEST


  Epoch   4/15 | val IoU 0.7844 | val F1 0.8792 | lr 1.00e-04 | 211s ✓ BEST


...

  Epoch  10/15 | val IoU 0.8242 | val F1 0.9036 | lr 1.00e-04 | 209s ✓ BEST


  Epoch  13/15 | val IoU 0.8294 | val F1 0.9068 | lr 1.00e-04 | 211s ✓ BEST


  Epoch  15/15 | val IoU 0.8437 | val F1 0.9152 | lr 1.00e-04 | 208s ✓ BEST



  TEST — IoU: 0.8436 | F1: 0.9152 | Prec: 0.9075 | Rec: 0.9229
  ✓ Kept checkpoint: /home/ubuntu/results_base_ablation/best_base_1_convnextl_unet_augmented.pth


NameError: name 'all_results' is not defined

In [ ]:
# Add ForensicNet reference (already trained by teammate)
all_results['forensicnet_reference'] = {
    'config_name': 'forensicnet_reference',
    'config': {'use_fpn': True, 'use_cbam': True, 'use_noise': True},
    'test_metrics': {'pixel_accuracy': 0.9931, 'precision': 0.9252, 'recall': 0.9108,
                     'f1': 0.9179, 'iou': 0.8483, 'dice': 0.9179},
    'total_params': 199560000, 'model_size_mb': 761.3,
    'note': 'Reference: 30 epochs, trained by teammate',
}

# Print cumulative table
print(f"\n{'='*90}")
print(f"  CUMULATIVE BASE ABLATION — ConvNeXt-L Components")
print(f"{'='*90}")
print(f"{'Config':<40} {'FPN':>4} {'CBAM':>5} {'Noise':>6} | "
      f"{'Test IoU':>8} {'Test F1':>8} {'Params':>8}")
print(f"{'─'*90}")

display_order = [
    'base_1_convnextl_unet',
    'base_1_convnextl_unet_augmented',
    'base_2_convnextl_fpn',
    'base_3_convnextl_fpn_cbam',
    'base_4_forensicnet_reproduction',
    'forensicnet_reference',
]

for name in display_order:
    if name not in all_results:
        continue
    r = all_results[name]
    c = r['config']
    tm = r['test_metrics']
    aug = '(+aug)' if 'augmented' in name else ''
    ref = '(ref 30ep)' if 'reference' in name else ''
    label = f"{name} {aug}{ref}"

    print(f"{label:<40} "
          f"{'✓' if c.get('use_fpn') else '·':>4} "
          f"{'✓' if c.get('use_cbam') else '·':>5} "
          f"{'✓' if c.get('use_noise') else '·':>6} | "
          f"{tm['iou']:>8.4f} {tm['f1']:>8.4f} "
          f"{r.get('total_params', 0)/1e6:>7.1f}M")

print(f"{'='*90}")

# Save complete results
with open(SAVE_DIR / 'base_ablation_complete.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

# Bar chart
names_plot = []
ious_plot = []
colors_plot = []

for name in display_order:
    if name not in all_results:
        continue
    r = all_results[name]
    short = name.replace('base_', '').replace('convnextl_', '')
    if 'reference' in name:
        short = 'ForensicNet (30ep ref)'
    names_plot.append(short)
    ious_plot.append(r['test_metrics']['iou'])
    if 'reference' in name:
        colors_plot.append('tab:red')
    elif 'augmented' in name:
        colors_plot.append('tab:green')
    else:
        colors_plot.append('tab:blue')

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(names_plot, ious_plot, color=colors_plot, edgecolor='black', linewidth=0.5)
for bar, iou in zip(bars, ious_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{iou:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Test IoU', fontsize=12)
ax.set_title('Base Ablation — ConvNeXt-L Component Contributions', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'base_ablation_chart.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"\n✓ Saved base_ablation_complete.json")
print(f"✓ Saved base_ablation_chart.png")
print(f"\nCheckpoints kept for XAI:")
for name in all_results:
    ckpt = SAVE_DIR / f'best_{name}.pth'
    if ckpt.exists():
        print(f"  {ckpt}")